# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**I chose a line chart because the question is about population growth over time. It allows the population trend of each continent to be compared from 1970 to 2022. I used an index starting at 100 in 1970 so that the growth of different continents can be compared fairly.**

In [3]:
# Q1
# YOUR CODE HERE

# Calculate total population for each continent and year
trend = (
    df_long.groupby(["Year", "Continent"])["Population"]
    .sum()
    .reset_index()
)

# Calculate growth relative to 1970
trend["Index (1970=100)"] = (
    trend["Population"]
    / trend.groupby("Continent")["Population"].transform("first")
    * 100
)

# Find the fastest-growing continent
latest = (
    trend[trend["Year"] == 2022]
    .set_index("Continent")["Index (1970=100)"]
)

winner = latest.idxmax()
growth = latest.max() - 100

# Use grey for other continents and an accent colour for the winner
palette = {
    continent: "#9ca3af"
    for continent in trend["Continent"].unique()
}

palette[winner] = "#2563a6"

fig = px.line(
    trend,
    x="Year",
    y="Index (1970=100)",
    color="Continent",
    markers=True,
    color_discrete_map=palette,
    title=f"{winner} grew fastest: population up {growth:.0f}% since 1970"
)

fig.update_layout(
    template="plotly_white",
    legend_title_text="",
    xaxis_title="",
    yaxis_title="Population index (1970 = 100)"
)

fig.update_xaxes(
    showgrid=False,
    dtick=10
)

fig.update_yaxes(
    showgrid=False
)

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.


**I chose a horizontal bar chart because it makes it easy to compare the 15 countries with the largest population increases. Sorting the countries also makes the ranking clear. I used a different colour for African countries so their representation in the top 15 can be seen easily.**

In [4]:
# Q2
# YOUR CODE HERE

q2 = df.copy()

# Calculate population added between 1970 and 2022
q2["People added"] = (
    q2["2022 Population"] -
    q2["1970 Population"]
)

# Select the top 15 countries
q2 = q2.nlargest(15, "People added")

# Sort for the horizontal bar chart
q2 = q2.sort_values("People added")

# Highlight African countries
q2["Group"] = q2["Continent"].apply(
    lambda x: "Africa" if x == "Africa" else "Other continents"
)

africa_count = (q2["Continent"] == "Africa").sum()

fig = px.bar(
    q2,
    x="People added",
    y="Country",
    orientation="h",
    color="Group",
    text="People added",
    title=f"Top 15 population increases: {africa_count} are in Africa"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    showlegend=False,
    xaxis_title="Population added from 1970 to 2022",
    yaxis_title=""
)

fig.update_xaxes(
    showgrid=False,
    tickformat=","
)

fig.update_yaxes(showgrid=False)

fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**I chose a scatter plot because it is useful for examining the relationship between two numerical variables. Here, land area is compared with the 2022 population of each country. I used logarithmic axes because countries have very different land areas and population sizes.**

In [5]:
# Q3
# YOUR CODE HERE


correlation = (
    np.log10(df["Area"])
    .corr(np.log10(df["2022 Population"]))
)

fig = px.scatter(
    df,
    x="Area",
    y="2022 Population",
    color="Continent",
    hover_name="Country",
    log_x=True,
    log_y=True,
    title=f"Relationship between land area and population (correlation = {correlation:.2f})"
)

fig.update_traces(
    marker=dict(
        size=8,
        opacity=0.7
    )
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Land area (km², log scale)",
    yaxis_title="2022 Population (log scale)",
    legend_title=""
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**I chose a box plot because the question asks about the spread of population density. A box plot shows the median, variation and unusual values for each continent instead of only showing an average. I used a logarithmic scale because population density differs greatly between countries.**

In [6]:
# Q4
# YOUR CODE HERE


median_density = (
    df.groupby("Continent")["Density"]
    .median()
    .sort_values(ascending=False)
)

highest = median_density.index[0]

fig = px.box(
    df,
    x="Continent",
    y="Density",
    color="Continent",
    points="outliers",
    log_y=True,
    title=f"{highest} has the highest median population density"
)

fig.update_layout(
    template="plotly_white",
    showlegend=False,
    xaxis_title="Continent",
    yaxis_title="Population density (people per km²)"
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**I chose a choropleth map because this question has a geographical focus. A map makes it easy to see where population growth and population decline occurred around the world. The colour scale also makes the difference between positive and negative population change clear.**

In [7]:
# Q5
# YOUR CODE HERE

q5 = df.copy()

# Calculate percentage population change
q5["Change (%)"] = (
    (q5["2022 Population"] - q5["2000 Population"])
    / q5["2000 Population"]
    * 100
)

# Count countries that grew and shrank
grew = (q5["Change (%)"] > 0).sum()
shrank = (q5["Change (%)"] < 0).sum()

# Use the same range for positive and negative changes
limit = q5["Change (%)"].abs().max()

fig = px.choropleth(
    q5,
    locations="CCA3",
    locationmode="ISO-3",
    color="Change (%)",
    hover_name="Country",
    hover_data={
        "Change (%)": ":.1f",
        "CCA3": False
    },
    color_continuous_scale=[
        "#2563A6",
        "#F7F7F7",
        "#D97706"
    ],
    color_continuous_midpoint=0,
    range_color=(-limit, limit),
    title=f"{grew} countries grew and {shrank} countries shrank from 2000 to 2022"
)

fig.update_layout(
    template="plotly_white",
    coloraxis_colorbar_title="Change (%)",
    geo=dict(
        showframe=False,
        showcoastlines=False
    )
)

fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**I chose a treemap because it can display the hierarchy of continents and the countries within them in one chart. The size of each rectangle represents population, making it easy to compare the contribution of different countries and continents to the world population.**

In [8]:
# Q6
# YOUR CODE HERE

largest_continent = (
    df.groupby("Continent")["2022 Population"]
    .sum()
    .idxmax()
)

fig = px.treemap(
    df,
    path=["Continent", "Country"],
    values="2022 Population",
    color="Continent",
    title=f"{largest_continent} contains the largest share of the 2022 world population"
)

fig.update_traces(
    textinfo="label+percent parent"
)

fig.update_layout(
    template="plotly_white",
    margin=dict(
        t=60,
        l=10,
        r=10,
        b=10
    ),
    legend_title=""
)

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**I chose a 100% stacked bar chart because the question focuses on population shares rather than absolute population. Each bar represents the complete world population for a particular year, so the changing proportions of Africa, Asia, Europe and the rest of the world can be compared directly.**

In [9]:
# Q7# YOUR CODE HERE

q7 = df.copy()

# Keep Africa, Asia and Europe separate
# Combine the remaining continents
q7["Region"] = q7["Continent"].where(
    q7["Continent"].isin(["Africa", "Asia", "Europe"]),
    "Rest of the world"
)

# Convert 1970 and 2022 into long format
q7 = q7.melt(
    id_vars="Region",
    value_vars=[
        "1970 Population",
        "2022 Population"
    ],
    var_name="Year",
    value_name="Population"
)

# Extract the year
q7["Year"] = q7["Year"].str[:4].astype(int)

# Calculate population by region
q7 = (
    q7.groupby(["Year", "Region"])["Population"]
    .sum()
    .reset_index()
)

# Calculate each region's share of the world population
q7["Share (%)"] = (
    q7["Population"]
    / q7.groupby("Year")["Population"].transform("sum")
    * 100
)

# Calculate the change in share
share_1970 = q7[q7["Year"] == 1970].set_index("Region")["Share (%)"]
share_2022 = q7[q7["Year"] == 2022].set_index("Region")["Share (%)"]

change = share_2022 - share_1970
rising_region = change.idxmax()

fig = px.bar(
    q7,
    x="Year",
    y="Share (%)",
    color="Region",
    barmode="stack",
    category_orders={
        "Region": [
            "Africa",
            "Asia",
            "Europe",
            "Rest of the world"
        ]
    },
    color_discrete_map={
        "Africa": "#D97706",
        "Asia": "#2563A6",
        "Europe": "#6B7280",
        "Rest of the world": "#CBD5E1"
    },
    title=f"{rising_region} had the largest increase in world population share"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Year",
    yaxis_title="Share of world population (%)",
    legend_title=""
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(
    showgrid=False,
    range=[0, 100]
)

fig.show()
